# 🐍 Python from Scratch — Module 6: Object-Oriented Programming (Classes)

### Grouping data and behavior into one thing

In Module 3 we built a store inventory as a dictionary of dictionaries. It worked, but
at larger scale it gets awkward — hard to add validation, hard to add a "compute total
value" method without a separate function sitting somewhere nearby. Classes are a way
to keep data (attributes) and the functions that work on it (methods) together, in one
self-contained thing.

## Table of Contents

1. [Why object-oriented programming](#sec1)
2. [`class`, `__init__`, and attributes](#sec2)
3. [Methods — the `self` parameter](#sec3)
4. [Multiple instances of the same class](#sec4)
5. [Class attributes vs instance attributes](#sec5)
6. [`__str__` — printing an object nicely](#sec6)
7. [The `_` and `__` convention — encapsulation](#sec7)
8. [Inheritance](#sec8)
9. [Overriding methods and polymorphism](#sec9)
10. [Fun fact: in Python, literally everything is an object](#sec10)
11. [Module summary](#sec11)
12. [Exercises](#sec12)

---

<a id="sec1"></a>
## 1. Why object-oriented programming

Remember the inventory exercise from Module 3:
`{"bread": {"price": 4.5, "qty": 20}}`. To compute a single product's value, you had to
remember that this particular dictionary has keys `price` and `qty` — nothing in the
code enforced that, so a typo or mix-up was easy. A class lets you say outright: "a
product is something that **always** has a price and a quantity, and **knows how** to
compute its own value" — data and logic become one coherent thing.

<a id="sec2"></a>
## 2. `class`, `__init__`, and attributes

A class is defined with the `class` keyword. The `__init__` method (the so-called
**constructor**) runs automatically whenever a new object (instance) is created, and is
used to set up the initial **attributes** — data attached to that specific object.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

kamil = Person("Kamil", 30)
print(kamil.name)
print(kamil.age)
print(type(kamil))

> 💡 **Fun fact**
>
> `__init__` is one of the so-called «dunder» methods (from *double underscore*). Python has several more (`__str__`, `__len__`, `__eq__`...) - special methods Python calls automatically in specific situations, like when an object is created, printed, or compared.

<a id="sec3"></a>
## 3. Methods — the `self` parameter

A function defined inside a class is a **method**. Its first parameter is always
`self` — a reference to the specific object the method was called on. The name `self`
is just a **convention** (you could technically call it something else), but stick with
it — the entire Python world uses it.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"Hi, I'm {self.name} and I'm {self.age} years old"

    def is_adult(self):
        return self.age >= 18

kamil = Person("Kamil", 30)
print(kamil.introduce())
print(kamil.is_adult())

> ⚠️ **Don't forget `self`**
>
> Leaving out `self` as a method's first parameter (`def introduce():` instead of `def introduce(self):`) will raise an error when you call `kamil.introduce()` - Python automatically passes the object as the first argument, so the method needs room for it in its parameter list.

<a id="sec4"></a>
## 4. Multiple instances of the same class

A class is a **template** (a blueprint), and an instance is a **specific object**
created from that template. A single class can produce any number of independent
instances, each with its own attribute values.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"Hi, I'm {self.name} and I'm {self.age} years old"

kamil = Person("Kamil", 30)
ania = Person("Ania", 25)

print(kamil.introduce())
print(ania.introduce())
print(kamil.name == ania.name)   # False - these are two independent objects

<a id="sec5"></a>
## 5. Class attributes vs instance attributes

An attribute defined **inside `__init__`** (via `self.something = ...`) belongs to a
specific instance — every object gets its own value. An attribute defined **directly in
the class body** (outside any method) is shared across every instance of that class.

In [ ]:
class Person:
    species = "Homo sapiens"   # class attribute - shared by every instance

    def __init__(self, name):
        self.name = name         # instance attribute - unique to each object

kamil = Person("Kamil")
ania = Person("Ania")

print(kamil.species, ania.species)   # same for both

Person.species = "Homo sapiens sapiens"   # change at the class level
print(kamil.species, ania.species)   # changed for both at once

> 💡 **A typical use for class attributes**
>
> A common pattern is a counter of how many objects have been created - a class attribute starts at 0, and every `__init__` call increments it (`Person.count += 1`). You'll see this in the exercises.

<a id="sec6"></a>
## 6. `__str__` — printing an object nicely

By default, `print(obj)` prints something unreadable, like
`<__main__.Person object at 0x...>`. The `__str__` method lets you decide exactly what
gets shown — Python calls it automatically for `print()` and `str()`.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"Person: {self.name} ({self.age} years old)"

kamil = Person("Kamil", 30)
print(kamil)          # without __str__ we'd see <__main__.Person object at ...>
print(str(kamil))

<a id="sec7"></a>
## 7. The `_` and `__` convention — encapsulation

Python doesn't have truly "private" attributes (like Java does), but it has a strong
convention: a single leading underscore (`self._attribute`) means "internal, don't touch
from outside", and a double leading underscore (`self.__attribute`) triggers so-called
*name mangling* — making accidental access from outside the class harder (though not
impossible).

In [ ]:
class BankAccount:
    def __init__(self, starting_balance):
        self._balance = starting_balance   # convention: "don't touch directly"

    def deposit(self, amount):
        self._balance += amount

    def show_balance(self):
        return self._balance

account = BankAccount(1000)
account.deposit(500)
print(account.show_balance())

# Technically still possible, but breaks the convention:
print(account._balance)

> ⚠️ **It's a convention, not a lock**
>
> A single leading underscore doesn't technically block anything - `account._balance = 999999` runs without an error. It's an agreement between programmers: "this is an implementation detail, change it through methods (`deposit`, `withdraw`), not directly." Breaking this convention is not a syntax error, but it is bad practice.

<a id="sec8"></a>
## 8. Inheritance

A class can **inherit** from another one — it then picks up all of that class's
attributes and methods, and can add its own or override existing ones. The class being
inherited from is called the **base** (or parent) class, and the one doing the
inheriting is the **derived** class (or subclass).

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def make_sound(self):
        return "..."

    def describe(self):
        return f"{self.name} says: {self.make_sound()}"


class Dog(Animal):   # Dog inherits from Animal
    def make_sound(self):   # overrides the base class's method
        return "Woof woof!"


class Cat(Animal):
    def make_sound(self):
        return "Meow!"

rex = Dog("Rex")
whiskers = Cat("Whiskers")

print(rex.describe())
print(whiskers.describe())

<a id="sec9"></a>
## 9. Overriding methods and polymorphism

The example above is already **polymorphism** — the same `describe()` method from the
base class works correctly for every subclass, even though `make_sound()` behaves
differently for each one. When a subclass wants to **extend**, rather than completely
replace, the base method, it uses `super()` to call the original implementation.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)   # calls Animal's __init__
        self.breed = breed         # plus something Animal doesn't have

rex = Dog("Rex", "German Shepherd")
print(rex.name, rex.breed)

<a id="sec10"></a>
## 10. Fun fact: in Python, literally everything is an object

Numbers, strings, functions, even classes themselves — they're all objects of some
class. `type()` (which you know from Module 1) is really showing you a value's class.

In [ ]:
print(type(5))            # <class 'int'>
print(type("text"))       # <class 'str'>
print(type([1, 2, 3]))    # <class 'list'>
print(type(print))        # <class 'builtin_function_or_method'>

print(isinstance(5, int))     # checks whether an object is an instance of a class
print(isinstance("a", int))

> 💡 **Fun fact**
>
> This is why you can call methods on numbers and strings (`"text".upper()`, `(5).bit_length()`) - they aren't some special built-in exception to the rule, they're ordinary objects of the `str` and `int` classes, exactly like your own `Person` class.

<a id="sec11"></a>
## 11. Module summary

By now it should be clear:

- how to define a class (`class`), a constructor (`__init__`), and methods,
- what `self` is and why it must be a method's first parameter,
- the difference between an instance attribute and a class attribute,
- how `__str__` works and what it's for,
- the `_`/`__` convention as a signal for "don't touch directly",
- how inheritance, `super()`, and method overriding work.

Object-oriented programming is a big topic — here you learned the fundamentals that most
of Python's ecosystem (including data-analysis and AI libraries) is built on, so this is
a solid investment for whatever you learn next.

<a id="sec12"></a>
## 12. Exercises

A few tasks deliberately revisit earlier modules (validation with `raise`, lists) — this
time wrapped in classes.

> 📝 **Exercise 1: A Person class**
>
> Define a class `Person` with attributes `name` and `job` (set in `__init__`) and a method `introduce()`, returning a sentence like: "Hi, I'm Kamil and I work as a developer." Create one instance and call the method.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class Person:
    def __init__(self, name, job):
        self.name = name
        self.job = job

    def introduce(self):
        return f"Hi, I'm {self.name} and I work as a {self.job}"

kamil = Person("Kamil", "developer")
print(kamil.introduce())
```
</details>

> 📝 **Exercise 2: Several instances at once**
>
> Define a class `Dog` with a `name` attribute and a method `bark()`, returning `f"{name} says: Woof!"`. Create three different instances (dogs with different names) and call `bark()` on each one in a `for` loop.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class Dog:
    def __init__(self, name):
        self.name = name

    def bark(self):
        return f"{self.name} says: Woof!"

dogs = [Dog("Rex"), Dog("Buddy"), Dog("Max")]

for dog in dogs:
    print(dog.bark())
```
</details>

> 📝 **Exercise 3: Instance counter (class attribute)**
>
> Define a class `User` with a class attribute `count = 0` that increases by 1 in `__init__` every time a new object is created. Create 3 instances and print `User.count` - it should show 3.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class User:
    count = 0

    def __init__(self, name):
        self.name = name
        User.count += 1

u1 = User("Kamil")
u2 = User("Ania")
u3 = User("Tomek")

print(User.count)   # 3
```

Hint: the update has to go through the class name (`User.count += 1`), not `self.count += 1` - the latter would create a new instance attribute instead of changing the shared counter.
</details>

> 📝 **Exercise 4: Printing nicely — `__str__`**
>
> Define a class `Product` with attributes `name` and `price`, and a `__str__` method that returns e.g. `"Bread - $4.50"` (price formatted to 2 decimal places). Create an object and print it with `print()`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        return f"{self.name} - ${self.price:.2f}"

bread = Product("Bread", 4.5)
print(bread)
```
</details>

> 📝 **Exercise 5: A bank account with validation**
>
> Define a class `BankAccount` with an attribute `_balance` (starting at 0) and methods `deposit(amount)` and `withdraw(amount)`. `withdraw` should raise a `ValueError` (`raise`) if you try to withdraw more than the current balance. Test both cases - a successful withdrawal and one that should raise an error (inside a `try`/`except` block).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class BankAccount:
    def __init__(self):
        self._balance = 0

    def deposit(self, amount):
        self._balance += amount

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

account = BankAccount()
account.deposit(1000)
account.withdraw(300)
print(account._balance)   # 700

try:
    account.withdraw(10000)
except ValueError as error:
    print(f"Error: {error}")
```
</details>

> 📝 **Exercise 6: Inheritance — shapes**
>
> Define a base class `Shape` with a method `area()` returning `0` (a placeholder). Define subclasses `Rectangle(Shape)` (with attributes `side_a`, `side_b`) and `Circle(Shape)` (with attribute `radius`), each overriding `area()` with its own formula (for a circle: `3.14159 * radius ** 2`). Create one instance of each and print their areas.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class Shape:
    def area(self):
        return 0

class Rectangle(Shape):
    def __init__(self, side_a, side_b):
        self.side_a = side_a
        self.side_b = side_b

    def area(self):
        return self.side_a * self.side_b

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

rectangle = Rectangle(4, 5)
circle = Circle(3)

print(f"Rectangle area: {rectangle.area()}")
print(f"Circle area: {circle.area():.2f}")
```
</details>

> 📝 **Exercise 7: Inventory as a class (Module 3 refactor)**
>
> Define a class `Product` (attributes `name`, `price`, `qty`, method `value()` returning `price * qty`) and a class `Inventory` with a list of products (attribute `products = []` in `__init__`), a method `add(product)`, and a method `total_value()` that sums up `value()` across all products. Add 3 products and print the total value.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class Product:
    def __init__(self, name, price, qty):
        self.name = name
        self.price = price
        self.qty = qty

    def value(self):
        return self.price * self.qty

class Inventory:
    def __init__(self):
        self.products = []

    def add(self, product):
        self.products.append(product)

    def total_value(self):
        total = 0
        for product in self.products:
            total += product.value()
        return total

inventory = Inventory()
inventory.add(Product("Bread", 4.5, 20))
inventory.add(Product("Milk", 3.2, 15))
inventory.add(Product("Eggs", 12.0, 8))

print(f"Total value: ${inventory.total_value():.2f}")
```

Notice how much more readable this is than the dictionary-of-dictionaries from Module 3 - `product.value()` says exactly what it does, and validation (e.g. that price can't be negative) could easily be added in one place, inside `__init__`.
</details>

> 🔥 **Exercise 8 (challenge): Mini library system**
>
> Define a class `Book` (attributes `title`, `available = True`) and a class `Library` with a list of books and methods: `add_book(book)`, `borrow(title)` (finds the book by title, sets `available = False`, and raises a `ValueError` with a sensible message if the book doesn't exist or is already borrowed), and `show_available()` (prints the titles of every available book). Add 3 books, borrow one, try to borrow it again (catch the error), and finally show what's available.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class Book:
    def __init__(self, title):
        self.title = title
        self.available = True

class Library:
    def __init__(self):
        self.books = []

    def add_book(self, book):
        self.books.append(book)

    def borrow(self, title):
        for book in self.books:
            if book.title == title:
                if not book.available:
                    raise ValueError(f"'{title}' is already borrowed")
                book.available = False
                return
        raise ValueError(f"Book not found: '{title}'")

    def show_available(self):
        for book in self.books:
            if book.available:
                print(book.title)

library = Library()
library.add_book(Book("The Witcher"))
library.add_book(Book("Solaris"))
library.add_book(Book("Dune"))

library.borrow("Dune")

try:
    library.borrow("Dune")
except ValueError as error:
    print(f"Error: {error}")

print("Available books:")
library.show_available()
```

Hint: this is the natural meeting point of classes (data + behavior), lists (the book collection), and the error handling from Module 5 (`raise ValueError` when trying to borrow an unavailable book) - every module in this series shows up in one exercise.
</details>

---

### What's next?

If the mini library system went reasonably smoothly, you have a solid grounding in
Python's object-oriented programming. Natural next steps from here: `dataclasses` (a
shortcut for writing simple data-container classes), working with external libraries
(most of which are themselves built out of classes), or a real project that ties all six
modules in this series together.